<div style="font-family:monospace; font-size:15px; line-height:1.5; border-top: 1px solid black; border-bottom: 1px solid black; padding: 10px; text-align: center;">
    <strong>DIANO - MANY-TO-ONE FUNCTIONAL MAPPING</strong><br>
    <strong></strong> SIVA VIKNESH<br>
    <strong></strong> siva.viknesh@sci.utah.edu / sivaviknesh14@gmail.com<br>
    <strong></strong> SCIENTIFIC COMPUTING & IMAGING INSTITUTE, UNIVERSITY OF UTAH, SALT LAKE CITY, UTAH, USA<br>
</div>

In [ ]:
# ****** IMPORTING THE NECESSARY LIBRARIES
import os
import torch
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from scipy.linalg import circulant, toeplitz, inv
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.parameter import Parameter
import vtk
from vtk.util.numpy_support import vtk_to_numpy 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.animation as animation

import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
processor = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("AVAILABLE PROCESSOR:", processor, '\n')

In [ ]:
seed = 10
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE PARAMETERS
</div>

In [ ]:
# ***** PARAMETERS FOR THE NEURAL NETWORK
input_var    = 4                                        # INPUT VARIABLES {a(x, y, z), x, y, z}
x_dim        = 128
y_dim        = 128
z_dim        = 128

# ***** HYPERPARAMETERS FOR THE NEURAL NETWORK
neurons      = 64
epochs       = 500
batchsize    = 2 
in_channels  = 4                       # NO. OF INPUT CHANNELS
mid_channels = 32                      # WIDTH OF THE CNN LAYERS IN THE SPECTRAL CONVOLUTION
out_channels = 1                       # NO. OF OUTPUT CHANNELS

learn_rate   = 1e-2
step_epoch   = 30
decay_rate   = 0.750

# ***** FILENAMES TO READ & WRITE THE DATA
mesh         = "Grid_128_0.vtk"
data_file    = "Grid_128_"
Nfile        = 100                                      # NO. OF TIME INSTANTS

# ***** LOCATION TO READ AND WRITE THE DATA
pwd          = os.getcwd()
os.chdir("../")
directory    = os.getcwd()
path         = directory + "/Flow_data/"
mesh         = path + mesh
data_file    = path + data_file

# ***** NORMALISATION OF FLOW VARIABLES
xscale   = 0.710591
yscale   = 0.332964
zscale   = 0.358831

Nbn      = 32
dt       = 0.01

h        = 1/Nbn
rho      = 1.06

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    READING THE DATA FILES
</div>


In [ ]:
''' ***************************************** MESH FILE ****************************************************** '''
print ("*"*85)
print ('READING THE MESH FILE: ', mesh[len(directory)+1:])
reader = vtk.vtkStructuredPointsReader()
reader.SetFileName(mesh)
reader.Update()
data = reader.GetOutput()
n_points = data.GetNumberOfPoints()
print ('NO. OF GRID POINTS IN THE MESH:', n_points)
x_vtk_mesh = np.zeros((n_points,1))
y_vtk_mesh = np.zeros((n_points,1))
z_vtk_mesh = np.zeros((n_points,1))
for i in range(n_points):
    pt_iso  =  data.GetPoint(i)
    x_vtk_mesh[i] = pt_iso[0] 
    y_vtk_mesh[i] = pt_iso[1]
    z_vtk_mesh[i] = pt_iso[2]
    
x  = np.reshape(x_vtk_mesh, (x_dim, y_dim, z_dim))/xscale 
y  = np.reshape(y_vtk_mesh, (x_dim, y_dim, z_dim))/yscale
z  = np.reshape(z_vtk_mesh, (x_dim, y_dim, z_dim))/zscale

print ("SHAPE OF X:",   x.shape)
print ("SHAPE OF Y:",   y.shape)
print ("SHAPE OF Z:",   z.shape)

print ("*"*85)

''' *************************************** FLOW FIELD FILE *************************************************** '''
fieldname1  = 'u_vel'                                                      # FIELD NAME FOR VTK FILES
fieldname2  = 'v_vel'
fieldname3  = 'w_vel'
fieldname4  = 'pressure'

u_vel     = np.zeros((Nfile, x_dim, y_dim, z_dim), dtype=np.float16)
v_vel     = np.zeros((Nfile, x_dim, y_dim, z_dim), dtype=np.float16)
w_vel     = np.zeros((Nfile, x_dim, y_dim, z_dim), dtype=np.float16)
Pressure  = np.zeros((Nfile, x_dim, y_dim, z_dim), dtype=np.float16)

for i in range(Nfile):
    file_name = data_file + str(i*4) + ".vtk"
    print ('READING THE DATA FILE: ', file_name[len(directory)+1:])
    reader = vtk.vtkStructuredPointsReader()
    reader.SetFileName(file_name)
    reader.Update()
    data = reader.GetOutput()
    
    pointData1 = data.GetPointData().GetArray(fieldname1)
    pointData2 = data.GetPointData().GetArray(fieldname2)    
    pointData3 = data.GetPointData().GetArray(fieldname3)    
    pointData4 = data.GetPointData().GetArray(fieldname4)   
    
    u_vel    [i, :, :, :] = np.reshape(vtk_to_numpy(pointData1), (x_dim, y_dim, z_dim))
    v_vel    [i, :, :, :] = np.reshape(vtk_to_numpy(pointData2), (x_dim, y_dim, z_dim))
    w_vel    [i, :, :, :] = np.reshape(vtk_to_numpy(pointData3), (x_dim, y_dim, z_dim))
    Pressure [i, :, :, :] = np.reshape(vtk_to_numpy(pointData4), (x_dim, y_dim, z_dim))
    
    print ("*"*85)

In [ ]:
umin, umax = np.min(u_vel), np.max(u_vel)
vmin, vmax = np.min(v_vel), np.max(v_vel)
wmin, wmax = np.min(w_vel), np.max(w_vel)
pmin, pmax = np.min(Pressure), np.max(Pressure)

Vmag = np.sqrt(u_vel[0, :, :, :]**2 + v_vel[0, :, :, :]**2 + w_vel[0, :, :, :]**2)
Vmag_sub = Vmag[::4, ::4, ::4]      # Compression Ratio is 4

# Get boolean mask where Vmag < 1e-3
mask = Vmag_sub < 1e-4

# Convert to array of indices: shape (N_points, 4)
index = torch.tensor(np.transpose(np.nonzero(mask)), dtype=torch.int, device=processor)

mask = torch.ones((1, Nbn, Nbn, Nbn, 1), dtype=torch.float32, device=processor)
mask[0, index[:, 0], index[:, 1], index[:, 2], 0] = 0.0
del Vmag, Vmag_sub, index

u_vel = u_vel/umax
v_vel = v_vel/vmax
w_vel = w_vel/wmax
Pressure = Pressure/pmax

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE CLASS FUNCTIONS
</div>


In [ ]:
def COMPLEX_MULTIPLY(input_data, weights):                             # COMPLEX WEIGHTS X FFT MODES 

    return torch.einsum("bixyz,ioxyz->boxyz", input_data, weights)

class CNN_LAYERS_3D(nn.Module):
    def __init__(self, in_channel, mid_channel, out_channel):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv3d(in_channel, mid_channel, kernel_size=1),
            nn.GELU(),
            nn.Conv3d(mid_channel, out_channel, kernel_size=1)
        )

    def forward(self, x):
        return self.main(x)

class SPECTRAL_BLOCK_3D (nn.Module):
    def __init__(self, in_channels, out_channels, x_modes, y_modes, z_modes):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.scale = 1 / (in_channels + out_channels)
        self.modes1 = x_modes
        self.modes2 = y_modes
        self.modes3 = z_modes

        self.weights = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, x_modes, y_modes, z_modes, dtype=torch.cfloat)
        )

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[2, 3, 4])
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-3), x.size(-2), x.size(-1)//2 + 1,
                             dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2, :self.modes3] = COMPLEX_MULTIPLY(
            x_ft[:, :, :self.modes1, :self.modes2, :self.modes3], self.weights
        )
        x = torch.fft.irfftn(out_ft, s=(x.size(-3), x.size(-2), x.size(-1)), dim=[2, 3, 4])
        return x


class ENCODER (nn.Module):
    def __init__(self, input_var, x_modes, y_modes, z_modes, width, grid, mask):
        super().__init__()
        self.p = nn.Linear(input_var, width)

        self.conv0 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)
        self.conv1 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)
        self.conv2 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)
        self.conv3 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)

        self.mlp0 = CNN_LAYERS_3D(width, width, width)
        self.mlp1 = CNN_LAYERS_3D(width, width, width)
        self.mlp2 = CNN_LAYERS_3D(width, width, width)
        self.mlp3 = CNN_LAYERS_3D(width, width, width)

        self.w0 = nn.Conv3d(width, width, 1)
        self.w1 = nn.Conv3d(width, width, 1)
        self.w2 = nn.Conv3d(width, width, 1)
        self.w3 = nn.Conv3d(width, width, 1)

        self.q = CNN_LAYERS_3D(width, width * 4, 1)
        self.reduce = nn.AvgPool3d(kernel_size=2, stride=2)
        self.grid = grid
        self.mask = mask

    def forward(self, x):
        x = torch.cat([x, self.grid], dim=-1)        
        x = self.p(x)
        x = x.permute(0, 4, 1, 2, 3)

        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x = F.gelu(x1 + self.w0(x))
        x = self.reduce(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x = F.gelu(x1 + self.w1(x))
        x = self.reduce(x)

        #x1 = self.conv2(x)
        #x1 = self.mlp2(x1)
        #x = F.gelu(x1 + self.w2(x))
        #x = self.reduce(x)

        #x1 = self.conv3(x)
        #x1 = self.mlp3(x1)
        #x = F.gelu(x1 + self.w3(x))
        #x = self.reduce(x)

        x = self.q(x)
        x = x.permute(0, 2, 3, 4, 1) 
        
        # Apply spatial mask
        x = x * self.mask
        return  x

class DECODER(nn.Module):
    def __init__(self, x_modes, y_modes, z_modes, width):
        super().__init__()
        self.p = nn.Linear(1, width)

        self.upsamp1 = nn.ConvTranspose3d(width, width, 2, stride=2)
        self.upsamp2 = nn.ConvTranspose3d(width, width, 2, stride=2)
        self.upsamp3 = nn.ConvTranspose3d(width, width, 2, stride=2)

        self.conv0 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)
        self.conv1 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)
        self.conv2 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)
        self.conv3 = SPECTRAL_BLOCK_3D(width, width, x_modes, y_modes, z_modes)

        self.mlp0 = CNN_LAYERS_3D(width, width, width)
        self.mlp1 = CNN_LAYERS_3D(width, width, width)
        self.mlp2 = CNN_LAYERS_3D(width, width, width)
        self.mlp3 = CNN_LAYERS_3D(width, width, width)

        self.w0 = nn.Conv3d(width, width, 1)
        self.w1 = nn.Conv3d(width, width, 1)
        self.w2 = nn.Conv3d(width, width, 1)
        self.w3 = nn.Conv3d(width, width, 1)

        self.q = nn.Conv3d(width, 1, 1)  # final projection

    def forward(self, x):
        x = self.p(x)  # Linear layer, (batch, depth, height, width, width)
        x = x.permute(0, 4, 1, 2, 3)  # (batch, width, depth, height, width)

        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x = F.gelu(x1 + self.w0(x))
        x = self.upsamp1(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x = F.gelu(x1 + self.w1(x))
        x = self.upsamp2(x)

        #x1 = self.conv2(x)
        #x1 = self.mlp2(x1)
        #x = F.gelu(x1 + self.w2(x))
        #x = self.upsamp3(x)

        #x1 = self.conv3(x)
        #x1 = self.mlp3(x1)
        #x = F.gelu(x1 + self.w3(x))

        x = self.q(x)

        return x.permute(0, 2, 3, 4, 1)  # (batch, depth, height, width, 1)


In [ ]:
class CDS(nn.Module):  # FIRST DERIVATIVE
    def __init__(self, Nx, Ny, Nz, hx, hy, hz):
        super().__init__()
        self.Nx = Nx
        self.hx = hx
        self.Ny = Ny
        self.hy = hy
        self.Nz = Nz
        self.hz = hz
        
        # Register buffers for the first derivative matrices
        self.register_buffer("Dx", self.compute_cds_matrix(Nx, hx))
        self.register_buffer("Dy", self.compute_cds_matrix(Ny, hy))
        self.register_buffer("Dz", self.compute_cds_matrix(Nz, hz))

    def compute_cds_matrix(self, N, h):
        a11, a12, a13 = 1.0, 0.0, -1.0
        d = np.zeros(N)
        d[-1], d[0], d[1] = a11, a12, a13

        # Use scipy's circulant function to generate the first derivative matrix
        D = torch.tensor(circulant(d), dtype=torch.float32, device=processor)/ (2.0 * h)

        # Apply boundary conditions
        D[0, :] = 0.0
        D[0, :3] = torch.tensor([-3.0, 4.0, -1.0], dtype=torch.float32, device=processor) / (2.0 * h)
        D[-1, :] = 0.0
        D[-1, -3:] = torch.tensor([1.0, -4.0, 3.0], dtype=torch.float32, device=processor) / (2.0 * h)

        return D

    def forward(self, x):
        # Compute the first derivatives using matrix multiplication       
        ddx = torch.einsum('ij, jkm -> ikm', self.Dx, x)
        ddy = torch.einsum('ij, kjm -> kim', self.Dy, x)
        ddz = torch.einsum('ij, mnj -> mni', self.Dz, x)
        return ddx, ddy, ddz
    
class OUCS2(nn.Module):
    def __init__(self, Nx, Ny, Nz, hx, hy, hz):
        super().__init__()
        self.Nx = Nx
        self.hx = hx
        self.Ny = Ny
        self.hy = hy
        self.Nz = Nz
        self.hz = hz
        
        # Register buffers as circulant matrices
        self.register_buffer("Dx", self.compute_OUCS_matrix(Nx, hx))
        self.register_buffer("Dy", self.compute_OUCS_matrix(Ny, hy))
        self.register_buffer("Dz", self.compute_OUCS_matrix(Nz, hz))

    def compute_OUCS_matrix(self, N, h):
        d = np.zeros(N)
        
        a = -40.0
        p33 = 36.0
        p32 = p33 / 3.0 - a / 12.0
        p34 = p33 / 3.0 + a / 12.0

        d[1] = p32
        d[0] = p33
        d[-1] = p34
        
        D1 = torch.tensor(circulant(d), dtype=torch.float32, device=processor)
        
        d = np.zeros(N)
        
        q31 = -p33 / 36.0 + a / 72.0
        q32 = -7.0 * p33 / 9.0 + a / 9.0
        q33 = -a / 4.0
        q34 = 7.0 * p33 / 9.0 + a / 9.0
        q35 = p33 / 36.0 + a / 72.0

        beta1 = 0.020
        beta2 = 0.090

        d[-2] = q35
        d[-1] = q34
        d[0] = q33
        d[1] = q32
        d[2] = q31

        D2 = torch.tensor(circulant(d), dtype=torch.float32, device=processor) / h

        out = torch.matmul(torch.linalg.inv(D1), D2)

        # Modify specific sections of the output matrix
        out[0:2, :] = 0.0
        out[0, :3] = torch.tensor([-3.0, 4.0, -1.0], dtype=torch.float32, device=processor) / (2.0 * h)
        out[1, :5] = torch.tensor([((2.0 * beta1) / 3.0 - 1.0 / 3.0), -((8.0 * beta1) / 3.0 + 0.50), 4.0 * beta1 + 1.0, -(8.0 * beta1 / 3.0 + 1.0 / 6.0), 2.0 * beta1 / 3.0])

        out[-2:, :] = 0.0
        out[-2, -5:] = torch.tensor([(-2.0 * beta2 / 3.0, (8.0 * beta2 / 3.0 + 1.0 / 6.0), -(4.0 * beta2 + 1.0), ((8.0 * beta2) / 3.0 + 0.50), -((2.0 * beta2) / 3.0 - 1.0 / 3.0))])
        out[-1, -3:] = torch.tensor([1.0, -4.0, 3.0], dtype=torch.float32, device=processor) / (2.0 * h)

        return out

    def forward(self, x):
        ddx = torch.einsum('ij, jkm -> ikm', self.Dx, x)
        ddy = torch.einsum('ij, kjm -> kim', self.Dy, x)
        ddz = torch.einsum('ij, mnj -> mni', self.Dz, x)
        return ddx, ddy, ddz


    
class PPE_EQUATION(nn.Module):                    # PRESSURE POISSON EQUATION
    def __init__(self, N, h, rho, mask):
        super().__init__()
        self.N    = N
        self.h    = h
        self.rho  = rho
        self.mask = mask
        
        self.Dx   = OUCS2 (self.N, self.N,  self.N, self.h, self.h, self.h)
        
    def forward(self, Vx, Vy, Vz):
        batch_size, Nx, Ny, Nz, _ = Vx.shape
        device = Vx.device

        # Initialize pressure field
        Pressure = torch.zeros((batch_size, Nx, Ny, Nz, 1), device=device)

        # Initialize RHS (right-hand side) field
        RHS = torch.zeros((batch_size, Nx, Ny, Nz), device=device)

        # Loop over batch to compute derivatives separately
        for b in range(batch_size):
            # Extract single Vx, Vy, Vz field (without last dim)
            Vx_b = Vx[b, ..., 0]
            Vy_b = Vy[b, ..., 0]
            Vz_b = Vz[b, ..., 0]

            # Compute derivatives for this sample
            dudx, dudy, dudz = self.Dx(Vx_b)
            dvdx, dvdy, dvdz = self.Dx(Vy_b)
            dwdx, dwdy, dwdz = self.Dx(Vz_b)

            # Compute RHS for this batch element
            RHS[b] = -self.rho * (dudx**2 + dvdy**2 + dwdz**2 + 
                              2.0*dudy*dvdx + 2.0*dudz*dwdx + 2.0*dvdz*dwdy)

        # Expand RHS to match Pressure shape (add last dim)
        RHS = RHS.unsqueeze(-1)

        h2 = self.h ** 2

        # Jacobi iterations
        max_iterations = 100
        tolerance = 1e-6

        for _ in range(max_iterations):
            P_pad = torch.nn.functional.pad(Pressure[..., 0], (1,1,1,1,1,1), mode='constant', value=0)

            neighbor_sum = (
                P_pad[:, 2:, 1:-1, 1:-1] + P_pad[:, :-2, 1:-1, 1:-1] +
                P_pad[:, 1:-1, 2:, 1:-1] + P_pad[:, 1:-1, :-2, 1:-1] +
                P_pad[:, 1:-1, 1:-1, 2:] + P_pad[:, 1:-1, 1:-1, :-2]
            )

            Pressure_new = (neighbor_sum - h2 * RHS[..., 0]) / 6.0

            residual = torch.norm(Pressure_new - Pressure[..., 0]) / torch.norm(Pressure_new)
            Pressure[..., 0] = Pressure_new

            if residual.item() < tolerance:
                break

        return Pressure*self.mask

In [ ]:
# MESH SPATIAL DATA
x_grid = torch.Tensor(x).to(processor)
y_grid = torch.Tensor(y).to(processor)
z_grid = torch.Tensor(z).to(processor)
grid   = torch.stack([x_grid, y_grid, z_grid], dim=-1).to(processor)
grid   = grid.unsqueeze(0).repeat(batchsize, 1, 1, 1, 1)

del x_grid, y_grid, z_grid
    
# FUNCTION MAPPING DATA
u_vel     = torch.Tensor(u_vel).to(processor)
v_vel     = torch.Tensor(v_vel).to(processor)
w_vel     = torch.Tensor(w_vel).to(processor)
Pressure  = torch.Tensor(Pressure).to(processor)

# SORTING THE TRAINING DATA
shape         = u_vel.size()
total_samples = shape[0]

train_size    = int(0.8 * total_samples)
test_size     = total_samples - train_size

u_inp  = u_vel.reshape    (total_samples, x_dim, y_dim, z_dim, 1)
v_inp  = v_vel.reshape    (total_samples, x_dim, y_dim, z_dim, 1)
w_inp  = w_vel.reshape    (total_samples, x_dim, y_dim, z_dim, 1)
P_out  = Pressure.reshape (total_samples, x_dim, y_dim, z_dim, 1)

dataset               = TensorDataset(u_inp, v_inp, w_inp, P_out)
train_data, test_data = random_split(dataset, [train_size, test_size], generator=torch.manual_seed(seed))

train_loader   = DataLoader(train_data, batch_size=batchsize, shuffle=True)
test_loader    = DataLoader(test_data,  batch_size=2,        shuffle=False)

del u_inp, v_inp, w_inp, P_out, u_vel, v_vel, w_vel, Pressure, train_data, test_data, dataset

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    TRAINING DATA
</div>


In [ ]:
x_modes      = 8                      
y_modes      = 8                     
z_modes      = 8                    

u_encoder    = ENCODER  (input_var, x_modes, y_modes, z_modes, mid_channels, grid, mask).to(processor)
v_encoder    = ENCODER  (input_var, x_modes, y_modes, z_modes, mid_channels, grid, mask).to(processor)
w_encoder    = ENCODER  (input_var, x_modes, y_modes, z_modes, mid_channels, grid, mask).to(processor)
diff_eqn     = PPE_EQUATION  (Nbn, h, rho, mask).to(processor)
P_decoder    = DECODER  (x_modes, y_modes, z_modes, mid_channels).to(processor)

u_optim      = optim.Adam(u_encoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)
v_optim      = optim.Adam(v_encoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)
w_optim      = optim.Adam(w_encoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)
P_optim      = optim.Adam(P_decoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)

u_scheduler  = optim.lr_scheduler.StepLR(u_optim, step_size=step_epoch, gamma=decay_rate)
v_scheduler  = optim.lr_scheduler.StepLR(v_optim, step_size=step_epoch, gamma=decay_rate)
w_scheduler  = optim.lr_scheduler.StepLR(w_optim, step_size=step_epoch, gamma=decay_rate)
p_scheduler  = optim.lr_scheduler.StepLR(P_optim, step_size=step_epoch, gamma=decay_rate)

Loss_Data   = torch.empty(size=(epochs+1, 1))
loss_func   = nn.MSELoss()

for epoch in range (epochs+1):
    total_loss = 0.0
    for batch_idx, (uin, vin, win, pout) in enumerate(train_loader):
        u_enc, v_enc, w_enc  = u_encoder (uin), v_encoder (vin), w_encoder (win)
        march_output         = diff_eqn     (u_enc, v_enc, w_enc)
        decoder_output       = P_decoder    (march_output)
        batch_loss           = loss_func    (decoder_output, pout) 

        u_optim.zero_grad()
        v_optim.zero_grad()
        w_optim.zero_grad()
        P_optim.zero_grad()
        batch_loss.backward()

        with torch.no_grad():
            u_optim.step()
            v_optim.step()
            w_optim.step()
            P_optim.step()

        total_loss += batch_loss.detach()  

    N = batch_idx + 1
    Loss_Data   [epoch] = total_loss/N
        
    print('TOTAL AVERAGE LOSS, [EPOCH =', epoch,']')
    print('LOSS          :', Loss_Data[epoch].item())
    print('LEARNING RATE :', u_optim.param_groups[0]['lr'])
    print ("*"*85)
    
    u_scheduler.step()
    v_scheduler.step()
    w_scheduler.step()
    p_scheduler.step()

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    SAVING THE FILES
</div>

In [ ]:
os.chdir(pwd)
torch.save(u_encoder.state_dict(), "DIANO_U_ENCODER_DT.pt" )
torch.save(v_encoder.state_dict(), "DIANO_V_ENCODER_DT.pt" )
torch.save(w_encoder.state_dict(), "DIANO_W_ENCODER_DT.pt" )
torch.save(P_decoder.state_dict(), "DIANO_P_DECODER_DT.pt" )

torch.save(Loss_Data  [0:epoch], "Loss_DT.pt"  )

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        TESTING DATA
</div>


In [ ]:
# TEST ERROR
u_encoder.eval()
v_encoder.eval()
w_encoder.eval()
P_decoder.eval()

total_loss = 0.0

with torch.no_grad():  # <-- Moved outside the loop
    for batch_idx, (uin, vin, win, pout) in enumerate(train_loader):
        u_enc, v_enc, w_enc  = u_encoder (uin), v_encoder (vin), w_encoder (win)
        march_output         = diff_eqn     (u_enc, v_enc, w_enc)
        decoder_output       = P_decoder    (march_output)
        batch_loss           = loss_func    (decoder_output, pout)
        total_loss += batch_loss

N = batch_idx + 1
Test_Loss = total_loss / N

print('TOTAL AVERAGE TESTING LOSS:', Test_Loss.item())
